# Knowledge Graph Entity Classification -- Wikidata5m

**Course:** KASDAD -- Universitas Indonesia, Fourth Semester  
**Group:** 5B  
**Date:** 2026-05-23

| No | Name | NPM |
|----|------|-----|
| 1 | Gunata Prajna Putra Sakri | 2406453461 |
| 2 | Melanton Gabriel Siregar | 2406365364 |
| 3 | Muhammad Vegard Fathul Islam | 2406365332 |
| 4 | Roben Joseph Buce Tambayong | 2406453594 |

## AI Disclosure

In accordance with assignment rules, we disclose the following AI tool usage:

| Tool | How it was used |
|------|-----------------|
| Claude (Anthropic) | Generating the filtering script skeleton, notebook structure, and .gitignore / README templates |

All code was reviewed, tested, and adapted by team members. Final analytical decisions, interpretations, and written sections are our own.

In [ ]:
import os
import json
import random

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

random.seed(42)
np.random.seed(42)
print('Imports OK')

## Section 1 -- Business Understanding

_[Fill in: describe the business/research problem, why entity classification on a knowledge graph matters, and what decisions this work could support.]_

In [ ]:
print('Section 1: Business Understanding')

## Section 2 -- Data Understanding

The dataset is derived from **Wikidata5m**, a large-scale knowledge graph containing ~5 million entities and ~20 million triples.  
We filter for five entity classes using the `P31` (instance-of) relation:

| Qcode | Class |
|-------|-------|
| Q5 | human |
| Q571 | book |
| Q532 | village |
| Q16521 | taxon |
| Q11424 | film |

Up to 2000 entities are sampled per class (random seed 42).

In [ ]:
FILTERED_CSV = os.path.join('data', 'raw', 'entities_filtered.csv')

df = pd.read_csv(FILTERED_CSV)
print('Shape:', df.shape)
print()
print('Class distribution:')
print(df.groupby(['class_qcode', 'class_name']).size().reset_index(name='count'))

### 2.2 Property Extraction

For each entity we collect every Wikidata predicate (P-code) that appears in the raw triples, **excluding P31** (instance-of) because that encodes the classification label.  
The extraction streams the multi-GB triples file line-by-line without loading it into RAM.  
Results are saved to `data/raw/entity_properties.json`.

> **To run once from the terminal:**  
> `python src/extract_properties.py --input wikidata5m_all_triplet.txt`

In [ ]:
import csv, json, os

TRIPLES_PATH  = "wikidata5m_all_triplet.txt"
ENTITIES_CSV  = os.path.join("data", "raw", "entities_filtered.csv")
OUTPUT_JSON   = os.path.join("data", "raw", "entity_properties.json")
LABEL_PRED    = "P31"   # instance-of — excluded because it IS the label
PROGRESS_STEP = 5_000_000

# --- load entity set ---
entity_set = set()
with open(ENTITIES_CSV, newline="", encoding="utf-8") as fh:
    for row in csv.DictReader(fh):
        entity_set.add(row["entity_id"])
print(f"Entities loaded: {len(entity_set):,}")

# --- stream triples: single pass, no full-RAM load ---
props = {eid: set() for eid in entity_set}
line_count = match_count = 0

# errors="replace" swaps bad bytes for U+FFFD instead of crashing
with open(TRIPLES_PATH, "r", encoding="utf-8", errors="replace") as fh:
    for line in fh:
        line_count += 1
        if line_count % PROGRESS_STEP == 0:
            print(f"  {line_count:,} lines ... {match_count:,} matches")
        parts = line.rstrip("\n").split("\t")
        if len(parts) != 3:
            continue
        subject, predicate, _ = parts
        # skip non-target entities and the label predicate
        if subject not in entity_set or predicate == LABEL_PRED:
            continue
        props[subject].add(predicate)
        match_count += 1

print(f"Done: {line_count:,} lines | {match_count:,} assignments")

# --- statistics ---
counts    = [len(v) for v in props.values()]
all_props = {p for codes in props.values() for p in codes}
entities_w = sum(1 for c in counts if c > 0)
print(f"\nUnique property codes : {len(all_props):,}")
print(f"Entities with props   : {entities_w:,} / {len(counts):,}")
print(f"Avg props per entity  : {sum(counts)/len(counts):.1f}")
print(f"Min / Max             : {min(counts)} / {max(counts)}")

# --- save: sets -> sorted lists for deterministic output ---
os.makedirs(os.path.dirname(OUTPUT_JSON), exist_ok=True)
serializable = {eid: sorted(codes) for eid, codes in props.items()}
with open(OUTPUT_JSON, "w", encoding="utf-8") as fh:
    json.dump(serializable, fh, separators=(",", ":"))
print(f"\nSaved: {OUTPUT_JSON}")

### 2.3 Feature Matrix Construction

We build a **binary sparse matrix** (scipy CSR format) where:
- Each **row** is one of our 10 000 entities (same order as `entities_filtered.csv`).
- Each **column** is a unique property P-code collected in 2.2 (P31 excluded).
- A cell is **1** if the entity has that property, **0** otherwise.

COO format is used as an intermediate step — we collect `(row, col)` pairs cheaply then convert to CSR in one shot, which is much faster than per-element insertion.

> **To run once from the terminal:**  
> `python src/build_feature_matrix.py`

In [ ]:
import csv, json, os
import numpy as np
import scipy.sparse as sp

ENTITIES_CSV = os.path.join("data", "raw", "entities_filtered.csv")
PROPS_JSON   = os.path.join("data", "raw", "entity_properties.json")
OUT_MATRIX   = os.path.join("data", "processed", "feature_matrix.npz")
OUT_COLUMNS  = os.path.join("data", "processed", "feature_matrix_columns.json")
OUT_LABELS   = os.path.join("data", "processed", "labels.csv")

# --- load entity order (defines row order of the matrix) ---
entity_order = []
with open(ENTITIES_CSV, newline="", encoding="utf-8") as fh:
    for row in csv.DictReader(fh):
        entity_order.append((row["entity_id"], row["class_qcode"], row["class_name"]))
print(f"Entities loaded: {len(entity_order):,}")

# --- load property sets ---
with open(PROPS_JSON, encoding="utf-8") as fh:
    props = json.load(fh)
print(f"Property JSON loaded: {len(props):,} entities")

# --- build sorted vocabulary (= column names) ---
all_prop_codes = {p for eid, _, _ in entity_order for p in props.get(eid, [])}
# P31 must not appear — it is the label, excluded during extraction
assert "P31" not in all_prop_codes, "P31 leaked into features!"
vocab = sorted(all_prop_codes)
print(f"Vocabulary size: {len(vocab):,} unique properties")

# --- build COO data then convert to CSR ---
prop_to_idx = {p: i for i, p in enumerate(vocab)}
row_idx_list, col_idx_list = [], []

for row_idx, (entity_id, _, _) in enumerate(entity_order):
    for prop in props.get(entity_id, []):
        col_idx = prop_to_idx.get(prop)
        if col_idx is not None:
            row_idx_list.append(row_idx)
            col_idx_list.append(col_idx)

data   = np.ones(len(row_idx_list), dtype=np.uint8)
matrix = sp.coo_matrix(
    (data, (row_idx_list, col_idx_list)),
    shape=(len(entity_order), len(vocab)),
).tocsr()

# sanity checks
assert matrix.shape[0] == len(entity_order)
assert matrix.shape[1] == len(vocab)
print(f"Matrix shape: {matrix.shape[0]:,} × {matrix.shape[1]:,}  ✓")

# --- statistics ---
nnz     = matrix.nnz
density = nnz / (matrix.shape[0] * matrix.shape[1]) * 100
mem_mb  = (matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes) / 1024**2
print(f"Non-zero entries : {nnz:,}")
print(f"Density          : {density:.4f}%")
print(f"In-memory (CSR)  : {mem_mb:.2f} MB")

# --- save outputs ---
os.makedirs(os.path.dirname(OUT_MATRIX), exist_ok=True)
sp.save_npz(OUT_MATRIX, matrix)

with open(OUT_COLUMNS, "w", encoding="utf-8") as fh:
    json.dump(vocab, fh, separators=(",", ":"))

with open(OUT_LABELS, "w", newline="", encoding="utf-8") as fh:
    writer = csv.writer(fh)
    writer.writerow(["entity_id", "class_name", "class_qcode"])
    for entity_id, class_qcode, class_name in entity_order:
        writer.writerow([entity_id, class_name, class_qcode])

print(f"\nSaved: {OUT_MATRIX}")
print(f"Saved: {OUT_COLUMNS}")
print(f"Saved: {OUT_LABELS}")

### 2.4 Property Name Lookup

We build two human-readable name dictionaries from the Wikidata5m alias files:

| File | Content |
|---|---|
| `wikidata5m_relation.txt` | 825 lines, each: `P-code TAB primary-alias TAB ...` |
| `wikidata5m_entity.txt` | 4.8M lines, each: `Q-code TAB primary-alias TAB ...` |

> **Data quality note:** the entity file's first alias for some of our target classes is unreliable (`Q5` → `'Huamn'`, `Q571` → `'wikipedia books/sandbox'`). We use hardcoded canonical names from `filter_entities.py` instead.

> **To run once from the terminal:**  
> `python src/build_property_names.py`

In [ ]:
import json, os

RELATIONS_FILE = "wikidata5m_relation.txt"
ENTITIES_FILE  = "wikidata5m_entity.txt"
OUT_PROPS      = os.path.join("data", "raw", "property_names.json")
OUT_CLASSES    = os.path.join("data", "raw", "class_names.json")

# Canonical names match filter_entities.py — NOT taken from the alias file
# because the entity file's first alias is noisy (e.g. 'Huamn' for Q5).
TARGET_CLASSES = {
    "Q5":     "human",
    "Q571":   "book",
    "Q532":   "village",
    "Q16521": "taxon",
    "Q11424": "film",
}

# --- load P-code -> primary alias from relation file ---
# Format: P-code TAB alias1 TAB alias2 ... (variable columns)
prop_names = {}
with open(RELATIONS_FILE, encoding="utf-8", errors="replace") as fh:
    for line in fh:
        parts = line.rstrip("\n").split("\t")
        if not parts[0].startswith("P"):
            continue
        p_code = parts[0]
        primary_alias = parts[1].strip() if len(parts) > 1 else ""
        prop_names[p_code] = primary_alias
print(f"Property codes loaded: {len(prop_names):,}")

# --- show raw entity aliases to illustrate data quality issue ---
remaining = set(TARGET_CLASSES)
raw_aliases = {}
with open(ENTITIES_FILE, encoding="utf-8", errors="replace") as fh:
    for line in fh:
        parts = line.rstrip("\n").split("\t")
        if parts[0] in remaining:
            raw_aliases[parts[0]] = parts[1:4]  # first 3 aliases only
            remaining.discard(parts[0])
        if not remaining:
            break

print("\nRaw first aliases from entity file (why we hardcode class names):")
print(f"  {"Q-code":<10} {"Alias[1]":<30} {"Alias[2]"}")
for qcode in sorted(TARGET_CLASSES):
    raw = raw_aliases.get(qcode, [])
    a1 = raw[0] if len(raw) > 0 else ""
    a2 = raw[1] if len(raw) > 1 else ""
    print(f"  {qcode:<10} {a1:<30} {a2}")

# --- save ---
os.makedirs(os.path.dirname(OUT_PROPS), exist_ok=True)
with open(OUT_PROPS, "w", encoding="utf-8") as fh:
    json.dump(prop_names, fh, ensure_ascii=False, indent=2)
with open(OUT_CLASSES, "w", encoding="utf-8") as fh:
    json.dump(TARGET_CLASSES, fh, ensure_ascii=False, indent=2)
print(f"\nSaved: {OUT_PROPS}  ({len(prop_names)} entries)")
print(f"Saved: {OUT_CLASSES}  ({len(TARGET_CLASSES)} entries)")

# --- sample lookup table for Gunata ---
print("\nSample property name lookups:")
for p, name in list(prop_names.items())[:10]:
    print(f"  {p:<10} {name}")

## Section 3 -- Exploratory Data Analysis (EDA)

In [ ]:
# (a) Class distribution bar chart
counts = df['class_name'].value_counts()
plt.figure(figsize=(8, 4))
sns.barplot(x=counts.index, y=counts.values)
plt.title('Entity count per class')
plt.xlabel('Class')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# (b) Property frequency heatmap -- placeholder
# TODO (Week 2): load the feature matrix and plot property frequencies per class
print('Property frequency heatmap -- to be implemented in Week 2')

In [ ]:
# (c) Sparsity analysis -- placeholder
# TODO (Week 2): compute sparsity of the feature matrix
print('Sparsity analysis -- to be implemented in Week 2')

## Section 4 -- Data Preparation

_To be filled in Week 2._

## Section 5 -- Modeling

_To be filled in Week 2._

## Section 6 -- Evaluation

_To be filled in Week 2._

## Section 7 -- Insights & Discussion

_To be filled in Week 3._